# `AnnotationPreprocessor.fetch_uniprot()`

`fetch_uniprot` queries the UniProt REST API for each `entry` and maps the returned feature array into the canonical `df_annot` schema (`protein_id`, `start`, `end`, `aa`, `feature_type`, `category`, `source`, `evidence`, `score`, `bond_id`). Bond features (disulfide, cross-link) expand to two endpoints sharing a `bond_id`; signal, propeptide and transit cleavage sites are anchored at the processing-span ends; `SITE` rows are routed by their description. `evidence='manual'` (default) keeps experimental and manually curated evidence and drops by-similarity annotations.

Requires `aaanalysis[pro]` (`requests`) and network access.

In [1]:
import warnings
import aaanalysis as aa
aa.options['verbose'] = False
warnings.filterwarnings('ignore')

# Three human proteins from the bundled gamma-secretase set (UniProt accessions as entries)
df_seq = aa.load_dataset(name='DOM_GSEC', n=10)
df_seq = df_seq[df_seq['entry'].isin(['Q14802', 'O43914', 'P01135'])].reset_index(drop=True)

annp = aa.AnnotationPreprocessor(verbose=False)
df_annot = annp.fetch_uniprot(df_seq=df_seq, features=['phospho', 'disulfide', 'binding', 'glyco_n'])
aa.display_df(df_annot, n_rows=10, show_shape=True)

DataFrame shape: (9, 10)


,protein_id,start,end,aa,feature_type,category,source,evidence,score,bond_id
1,P01135,47,47,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_1
2,P01135,60,60,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_1
3,P01135,55,55,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_2
4,P01135,71,71,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_2
5,P01135,73,73,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_3
6,P01135,82,82,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_3
7,O43914,50,50,D,binding,Functional sites,UniProt,ECO:0000269,1.000000,nan
8,O43914,35,35,C,disulfide,PTMs,UniProt,ECO:0000269,1.000000,O43914_disulfide_1
9,O43914,35,35,C,disulfide,PTMs,UniProt,ECO:0000269,1.000000,O43914_disulfide_1


`df_annot` feeds `encode`, which turns the annotations into a `[0, 1]`-normalized per-residue `dict_num` for `CPP.run_num`. Entries without any of the requested annotations are handled by `encode`'s `on_mismatch` policy.

In [2]:
dict_num = annp.encode(df_seq=df_seq, df_annot=df_annot, features=['disulfide', 'binding'], on_mismatch='drop')
print({entry: arr.shape for entry, arr in dict_num.items()})

{'Q14802': (87, 2), 'P01135': (160, 2), 'O43914': (113, 2)}


## Further parameters

`evidence` widens the evidence filter (`'experimental'` keeps only experimental evidence, `'manual'` adds manually curated evidence, `'all'` keeps everything, including by-similarity annotations); `timeout` bounds each REST request in seconds; `max_workers > 1` fetches on a thread pool with input-ordered, byte-identical results (opt-in, because parallel UniProt requests risk HTTP-429 throttling).

In [3]:
df_annot_all = annp.fetch_uniprot(df_seq=df_seq, features=['disulfide'], evidence='all',
                                  timeout=60.0, max_workers=2)
print('disulfide endpoints, manual evidence:', int((df_annot['feature_type'] == 'disulfide').sum()),
      '| all evidence:', len(df_annot_all))
aa.display_df(df_annot_all, n_rows=10, show_shape=True)

disulfide endpoints, manual evidence: 8 | all evidence: 8
DataFrame shape: (8, 10)


,protein_id,start,end,aa,feature_type,category,source,evidence,score,bond_id
1,P01135,47,47,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_1
2,P01135,60,60,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_1
3,P01135,55,55,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_2
4,P01135,71,71,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_2
5,P01135,73,73,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_3
6,P01135,82,82,C,disulfide,PTMs,UniProt,ECO:0000255,1.000000,P01135_disulfide_3
7,O43914,35,35,C,disulfide,PTMs,UniProt,ECO:0000269,1.000000,O43914_disulfide_1
8,O43914,35,35,C,disulfide,PTMs,UniProt,ECO:0000269,1.000000,O43914_disulfide_1
